# 02: Modeling

## Preliminares

In [14]:
# importar librerias
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime
import os
from src.functions import *
from sklearn.model_selection import train_test_split, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

In [15]:
# Abrir ficheros de datos limpios
clean_df = pd.read_csv('data/clean/merged_df_clean.csv')
y_full = pd.read_csv('data/clean/y_full.csv')
ids_competencia = pd.read_csv('data/clean/ids_test.csv')
with open('data/clean/len_train.txt', 'r') as f:
    len_train = int(f.read())

clean_df.shape, y_full.shape, ids_competencia.shape, len_train

((2919, 75), (1460, 2), (1459, 1), 1460)

## Modelado

Se aplicarán 2 modelos de predicción:
* `Random Forest Regressor`
* `LightGBM`

In [16]:
# Se crean listas de variables por tipo

# Numéricas
columnas_numericas = clean_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Columnas numéricas: {len(columnas_numericas)}")

# Categóricas
columnas_categoricas = clean_df.select_dtypes(include=['object', 'category', 'str']).columns.tolist()
print(f"\nColumnas categóricas: {len(columnas_categoricas)}")

# Muestreo Estratificado: Luego de la depuración, se separa el test de la competencia del train original
X_competencia = clean_df.iloc[len_train:].copy()
X = clean_df.iloc[:len_train].copy()

# Particionar training test
X_train, X_test, y_train, y_test = train_test_split(X,
                                                    y_full['SalePriceLog'],
                                                    test_size=0.2,
                                                    random_state=42
                                                    )
# Se inicializa el registro de métricas
log_resultados = []

Columnas numéricas: 32

Columnas categóricas: 43


Se aplica Random Forest Regressor como primer modelo:

In [17]:
# Pipeline modelo1
nombre_modelo1 = 'Random Forest'

# Definir el preprocesado
preprocesado_rf = ColumnTransformer(
    transformers=[
        # Las numéricas pasan tal cual en el RF
        ('num', 'passthrough', columnas_numericas),

        # One-Hot Encoder para categoricas (crea pocas columnas nuevas)
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), columnas_categoricas)
    ]
)

# Crear el pipeline
pipe_rf = Pipeline([
    ('pre', preprocesado_rf),
    ('rf', RandomForestRegressor(random_state=42,
                                 n_estimators=300,
                                 max_depth=40,
                                 min_samples_leaf=1,                              
                                 max_features='sqrt',
                                 max_samples=None                                  
                                 ))
])
# Entrenamiento
pipe_rf.fit(X_train, y_train)

# Predicción
y_pred = pipe_rf.predict(X_test)

# Se calculan las métricas y se guardan en log
log_resultados.append(obtener_metricas(y_test, y_pred, nombre_modelo1))
print("Entrenamiento finalizado.")
print("\nMétricas:", log_resultados[-1]) # se muestra siempre el último modelo en ejecutar

Entrenamiento finalizado.

Métricas: {'Modelo': 'Random Forest', 'MAE': 0.10712446260249141, 'MSE': 0.02650631404674232, 'RMSE': np.float64(0.16280759824634206), 'R2': 0.8579595218526144}


In [18]:
# Pipeline modelo2
nombre_modelo2 = 'LightGBM'
# Creamos el pipeline reutilizando el preprocesamiento del Random Forest
pipe_lgb = Pipeline([
    ('pre', preprocesado_rf),
    ('lgb', lgb.LGBMRegressor(random_state=42,
                              subsample=0.8,
                              colsample_bytree=0.8,
                              n_estimators = 500,
                              max_depth = 5,
                              learning_rate = 0.05,
                              num_leaves = 20,
                              child_samples = 20,
                              verbose=-1))
])
# Entrenamiento
pipe_lgb.fit(X_train, y_train)
# Predicción
y_pred_lgb = pipe_lgb.predict(X_test)
# Se calculan las métricas y se guardan en el log de resultados
log_resultados.append(obtener_metricas(y_test, y_pred_lgb, nombre_modelo2))
print("Entrenamiento de LightGBM finalizado.")
print("\nMétricas:", log_resultados[-1])

Entrenamiento de LightGBM finalizado.

Métricas: {'Modelo': 'LightGBM', 'MAE': 0.08859552717489481, 'MSE': 0.019541700239442426, 'RMSE': np.float64(0.13979163150719154), 'R2': 0.8952810850679398}


## Evaluación de Resultados

In [19]:
# Métricas obtenidas:

# Crear DataFrame comparativo
ahora = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
df_metricas = pd.DataFrame(log_resultados)
df_metricas['fecha_hora'] = ahora

# Se guarda en csv
ruta_log = 'data/log_metricas.csv'
file_exists = os.path.isfile(ruta_log) # comprobar si el fichero ya existe
df_metricas.to_csv(ruta_log, mode='a', index=False, header=not file_exists)

# Mostrar tabla
display(df_metricas.sort_values(by='RMSE', ascending=True).drop(columns='fecha_hora'))

,Modelo,MAE,MSE,RMSE,R2
1,LightGBM,0.088596,0.019542,0.139792,0.895281
0,Random Forest,0.107124,0.026506,0.162808,0.857960


Luego de haber ajustado los hiperparámetros, el modelo LightGBM obtuvo los menores errores en la predicción, y por lo tanto se escoje como modelo final.

Se reentrena el modelo final con los datos completos:

In [20]:
modelo_final = pipe_lgb
modelo_final.fit(X, y_full['SalePriceLog'])
print("Entrenamiento finalizado.")

Entrenamiento finalizado.


In [21]:
# Importancia de factores en el modelo
importances = modelo_final[-1].feature_importances_

# Se obtiene el último paso del pipeline y extrae las feature_importances_
preprocessor = modelo_final.named_steps['pre']
feature_names = preprocessor.get_feature_names_out()

feature_importance_df = pd.DataFrame({'feature': feature_names, 'importance': importances})
feature_importance_df = feature_importance_df.sort_values(by='importance', ascending=False)
feature_importance_df.head(20)

,feature,importance
7,num__GrLivArea,415
6,num__TotalBsmtSF,400
2,num__LotArea,399
25,num__AvgRoomSize,343
21,num__HouseAge,293
29,num__TotalPorchSF,261
23,num__RatioBsmt,258
22,num__RemodelAge,256
27,num__GarageAge,233
3,num__OverallQual,232


* Las columnas creadas en *Feature Engineering* dominan el ranking de variables, con `AvgRoomSize`, `HouseAge`, `TotalPorchSF`, `RatioBsmt`, `RemodelAge` y `GarageAge` en el top 10.
* Las 3 primeras características son puramente dimensionales.
* De las 43 variables categóricas iniciales que pasaron por el OneHotEncoder, creando nuevas columnas, solo una de ellas ingresó en el top 20: `cat__Condition1_Norm`. Esta variable indica que la condición de ubicación es "Normal" (no está pegada a una vía de tren ruidosa ni a una autopista principal).

Se evalua el modelo final usando CV de 5 particiones:

In [22]:
# Evaluar el modelo final usando CV de 5 particiones
scores = cross_validate(modelo_final, X, y_full['SalePriceLog'],
                        cv=5, scoring=['neg_root_mean_squared_error', 'neg_mean_absolute_error', 'r2'], n_jobs=-1)


# Multiplicar por menos (-) los errores para compensar el negativo de scikit-learn
rmse_medio = -scores['test_neg_root_mean_squared_error'].mean()
mae_medio = -scores['test_neg_mean_absolute_error'].mean()
r2_medio = scores['test_r2'].mean() # R2 se lee tal cual, no necesita negativo
# Mostrar los resultados
print(f"RMSE Medio (Validación Cruzada): {rmse_medio:.5f}")
print(f"MAE Medio (Validación Cruzada): {mae_medio:.5f}")
print(f"R2 Medio (Validación Cruzada): {r2_medio:.5f}")

RMSE Medio (Validación Cruzada): 0.12599
MAE Medio (Validación Cruzada): 0.08480
R2 Medio (Validación Cruzada): 0.89996


Se predice sobre los valores de la competencia de Kaggle y se crea el fichero de envío `data/submission.csv`: 

In [23]:
# Predecir sobre el dataset de la competencia
y_pred_final = modelo_final.predict(X_competencia)
# Se pasan de escala logarítmica a dólares normales
precios_finales = np.expm1(y_pred_final)
# Aseguramos de que todo sea plano 
ids_planos = np.ravel(ids_competencia) 
precios_planos = np.ravel(precios_finales)
# Se exporta
salida = pd.DataFrame({
    'Id': ids_planos, 
    'SalePrice': precios_planos
})
salida.to_csv('data/submission.csv', index=False)
print('Fichero de salida creado en la carpeta data del repositorio.')

Fichero de salida creado en la carpeta data del repositorio.


### Anexo: Optimización de Hiperparámetros

* La siguiente celda se utilizó para optimizar los hiperparámetros de los modelos de forma individual. La ejecución de la misma no es necesaria.
* En caso de utilizarla, reemplazar 'no' por 'si' en la primer línea. Puede demorar varios minutos.

In [24]:
ejecutar_celda = 'no' # Reemplazar para ejecutar
modelo_a_optimizar = 2  # 1 = Random Forest | 2 = LightGBM

if ejecutar_celda == 'si':

    if modelo_a_optimizar == 1:
        nombre_modelo= 'Random Forest'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para Random Forest
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocesado_rf),
            ('rf', RandomForestRegressor(random_state=42))
        ])

        param_grid = {
            'rf__n_estimators': [300],
            'rf__max_depth': [40],
            'rf__min_samples_leaf': [1],
            'rf__max_features': ['sqrt'],
            'rf__max_samples': [None]
        }

    elif modelo_a_optimizar == 2:
        nombre_modelo= 'LightGBM'
        print(f"Configurando GridSearchCV para {nombre_modelo}")

        # Pipeline usando el preprocesador específico para LightGBM
        modelo_base = Pipeline(steps=[
            ('preprocesador', preprocesado_rf),
            ('lgb', lgb.LGBMRegressor(random_state=42,
                                      subsample=0.8,
                                      colsample_bytree=0.8
))
        ])

        param_grid = {
            'lgb__n_estimators': [500, 550, 600],
            'lgb__max_depth': [5],
            'lgb__learning_rate': [0.05],
            'lgb__num_leaves': [20],
            'lgb__min_child_samples': [20]
        }

    else:
        raise ValueError("Opción no válida. Por favor, selecciona 1 o 2.")

    # Configurar el GridSearchCV
    grid_search = GridSearchCV(
        estimator=modelo_base,
        param_grid=param_grid,
        scoring='neg_root_mean_squared_error',
        cv=3,
        n_jobs=-1,
        verbose=2
    )

    # Entrenar con X_train y y_train originales
    print(f"Iniciando búsqueda de hiperparámetros. Esto tomará unos minutos.")
    grid_search.fit(X_train, y_train)

    # Resultados
    print("\n--- Búsqueda Finalizada ---")
    print("Mejores hiperparámetros encontrados:")
    print(grid_search.best_params_)

    # Extraer el mejor modelo
    best_model = grid_search.best_estimator_

    # Predecir en el set de prueba usando X_test original
    y_pred = best_model.predict(X_test)

    print("\nEvaluación en el set de Prueba:\n", obtener_metricas(y_test, y_pred, nombre_modelo))